# Feature Engineering and Model Training

Tang Bin — dataset preparation and model-training supplement to `01_data_exploration.ipynb`.

We use the repository's Kaggle `data/results.csv`. Prediction samples start in 1990; earlier matches only initialise histories. The last 20% by date is held out, with whole dates kept together. All model settings are fixed in advance: this is an exploratory benchmark, not independently validated hyperparameter selection.

Scores include extra time and exclude shootouts, so the target is not strictly a 90-minute result. Elo expectancy is an expected score (win=1, draw=0.5), not a three-class win probability.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src' / 'train.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'results.csv').exists()
sys.path.insert(0, str(ROOT / 'src'))
import train
raw = pd.read_csv(ROOT / 'data' / 'results.csv')
print('Raw rows:', len(raw))
print('Exact duplicate rows:', int(raw.duplicated().sum()))
print('Missing values:', raw.isna().sum().to_dict())
print('1990+ rows:', int((pd.to_datetime(raw.date) >= '1990-01-01').sum()))

Raw rows: 49945
Exact duplicate rows: 0
Missing values: {'date': 0, 'home_team': 0, 'away_team': 0, 'home_score': 0, 'away_score': 0, 'tournament': 0, 'city': 0, 'country': 0, 'neutral': 0}
1990+ rows: 32821


## Pre-match predictors

Historical win rate, goals for/against, goal difference, points per game, 5/10-match form, rest days, directional head-to-head history, Elo (initial 1500, K=30, home offset 60), tournament and neutral venue. All matches on a date use histories strictly before that date. Imputation, scaling and categorical encoding are fitted on training rows only. Historical results from earlier test dates update form and Elo: this simulates repeated match-day prediction, not a forecast of the whole test decade made on the cutoff date.

Missing history stays missing until train-only imputation. No score, scorer, shootout winner or outcome column enters the predictors.

In [2]:
import numpy as np
# Verify current/same-day/future scores cannot change earlier pre-match features.
fixture = pd.DataFrame({'date': pd.to_datetime(['2000-01-01','2000-01-01','2000-01-02']), 'source_row':[0,1,2], 'home_team':['A','A','A'], 'away_team':['B','C','B'], 'home_score':[1,2,0], 'away_score':[0,0,1], 'neutral':[False]*3, 'tournament':['Friendly']*3})
a = train.build_features(fixture)
changed = fixture.copy()
changed.loc[0, 'home_score'] = 9
b = train.build_features(changed)
pd.testing.assert_frame_equal(a.loc[:1, train.NUMERIC], b.loc[:1, train.NUMERIC])
changed = fixture.copy()
changed.loc[2, 'away_score'] = 8
b = train.build_features(changed)
pd.testing.assert_frame_equal(a[train.NUMERIC], b[train.NUMERIC])
assert a.loc[1, 'home_games'] == 0
assert a.loc[2, 'home_games'] == 2
print('Passed: own-score, same-day and future-score leakage checks.')

Passed: own-score, same-day and future-score leakage checks.


In [3]:
train.main()

Training: majority_class_dummy
Training: logistic_regression
Training: random_forest
Training: extra_trees
{
  "metadata": {
    "source": "Kaggle: martj42/international-football-results-from-1872-to-2017",
    "input_sha256": "0b67e6a268a329892d288e03fa90000fac7566022d7bca9c59b104c3704d5716",
    "run_utc": "2026-11-11T03:28:20.836101+00:00",
    "rows_raw": 49945,
    "rows_clean": 49945,
    "date_min": "1872-11-30",
    "date_max": "2026-10-06",
    "cutoff_date": "2019-11-19",
    "train_rows": 26212,
    "test_rows": 6609,
    "features": "pre-match form, rest, directional head-to-head, Elo, tournament, neutral venue",
    "model_rows": 32821,
    "model_start": "1990-01-01",
    "python": "3.12.14",
    "pandas": "3.0.1",
    "sklearn": "1.5.2",
    "evaluation": "Sequential pre-match prediction: earlier test outcomes update histories; model weights stay fixed.",
    "confusion_matrix_labels": [
      "home_win",
      "draw",
      "away_win"
    ]
  },
  "summary": {
    "majo

In [4]:
summary = pd.read_csv(ROOT / 'results' / 'model_metrics_summary.csv')
meta = json.loads((ROOT / 'results' / 'model_run_metadata.json').read_text())
print(json.dumps(meta, indent=2))
print(summary.to_string(index=False))

{
  "source": "Kaggle: martj42/international-football-results-from-1872-to-2017",
  "input_sha256": "0b67e6a268a329892d288e03fa90000fac7566022d7bca9c59b104c3704d5716",
  "run_utc": "2026-11-11T03:28:20.836101+00:00",
  "rows_raw": 49945,
  "rows_clean": 49945,
  "date_min": "1872-11-30",
  "date_max": "2026-10-06",
  "cutoff_date": "2019-11-19",
  "train_rows": 26212,
  "test_rows": 6609,
  "features": "pre-match form, rest, directional head-to-head, Elo, tournament, neutral venue",
  "model_rows": 32821,
  "model_start": "1990-01-01",
  "python": "3.12.14",
  "pandas": "3.0.1",
  "sklearn": "1.5.2",
  "evaluation": "Sequential pre-match prediction: earlier test outcomes update histories; model weights stay fixed.",
  "confusion_matrix_labels": [
    "home_win",
    "draw",
    "away_win"
  ]
}
               model  accuracy  balanced_accuracy  macro_f1
majority_class_dummy  0.477228           0.333333  0.215371
 logistic_regression  0.608413           0.509239  0.464762
       random_

In [5]:
reports = json.loads((ROOT / 'results' / 'model_metrics.json').read_text())
for name, report in reports.items():
    print('\n' + name)
    print('Confusion matrix: rows=true, columns=predicted; labels=', train.TARGETS)
    print(np.array(report['confusion_matrix']))
    print(pd.DataFrame(report['classification_report']).T.to_string())


majority_class_dummy
Confusion matrix: rows=true, columns=predicted; labels= ['home_win', 'draw', 'away_win']
[[3154    0    0]
 [1507    0    0]
 [1948    0    0]]
              precision    recall  f1-score      support
home_win       0.477228  1.000000  0.646113  3154.000000
draw           0.000000  0.000000  0.000000  1507.000000
away_win       0.000000  0.000000  0.000000  1948.000000
accuracy       0.477228  0.477228  0.477228     0.477228
macro avg      0.159076  0.333333  0.215371  6609.000000
weighted avg   0.227747  0.477228  0.308343  6609.000000

logistic_regression
Confusion matrix: rows=true, columns=predicted; labels= ['home_win', 'draw', 'away_win']
[[2772   47  335]
 [ 955   51  501]
 [ 706   44 1198]]
              precision    recall  f1-score      support
home_win       0.625310  0.878884  0.730724  3154.000000
draw           0.359155  0.033842  0.061856  1507.000000
away_win       0.588987  0.614990  0.601708  1948.000000
accuracy       0.608413  0.608413  0.60841

## Interpretation and limitations

Compare macro-F1 and draw recall alongside accuracy and the majority-class baseline. Algorithm rankings describe this holdout, not guaranteed future performance. Inspect draw errors and distinguish performance gains from class-weighting differences. No formal significance claim or exhaustive tuning is made.

The EDA examined this full period before the split; therefore the holdout is an initial chronological benchmark rather than a pristine blind final test. Before selecting or tuning a final model, use earlier rolling validation windows and reserve a fresh future period. Further improvements: Elo ablation, modern-history sensitivity, calibrated probabilities, and player/lineup data.

The teammate EDA and project webpage are preserved. Its full-history venue calculations and 1990+ summary use different populations; resolve that scope in the report before submission.